# FM-MCAN-HDD: End-to-End Federated Multimodal Heart-Disease Research Implementation

**Based on the uploaded paper:** *Trust-Aware Multimodal Privacy-Preserving Federated Learning Framework with Cross-Modal Attention for Early Heart Disease Diagnosis.*

This notebook implements a runnable research prototype covering:
1. Reproducibility/configuration and data validation
2. Synthetic aligned multimodal demo data (image-like cardiac inputs, ECG sequences, clinical features), partitioned into four simulated hospital clients
3. Image / temporal ECG / tabular encoders, cross-modal attention, and a clinical multi-task head
4. Local training, trust-score weighting, gradient/update clipping, Gaussian-noise perturbation, and federated rounds
5. Evaluation, confusion matrix, ROC-AUC, training curves, fold-wise demonstration, ablation-ready model options
6. Local explainability examples: image saliency/Grad-CAM-style map, ECG temporal attention, and clinical-feature gradients
7. Saving model, metrics, plots, and configuration

## Important scope and scientific integrity
The paper describes several components at a high level, but the uploaded manuscript does **not** provide the original source code, trained weights, complete patient-level data, or enough detail to reproduce its reported experiments exactly. Therefore this notebook is a **working, self-contained prototype** with synthetic demo data so every cell can run without private datasets. It does not validate the manuscript's reported 98.37% accuracy or 99.31% AUC. Do not report synthetic-demo metrics as clinical evidence.

The paper's dataset section describes CT/MRI images, ECG chart images, raw ECG records, and UCI-style clinical records. Real-world use requires correctly paired patient IDs/labels and locally stored datasets at each hospital. The synthetic demo uses aligned modalities for software testing. The simple update aggregation below simulates trust-weighted aggregation and DP noise; it is **not production cryptographic secure aggregation or a formal end-to-end differential-privacy guarantee**. Do not use for clinical decisions.

## 1. Install / import dependencies
Run in Google Colab or a Python environment with PyTorch, scikit-learn, pandas, matplotlib and seaborn. If a package is already installed, pip will leave it available.

In [ ]:
# Uncomment in a fresh Colab runtime if needed:
# %pip -q install torch torchvision scikit-learn pandas matplotlib seaborn

import os, json, copy, random, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             roc_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| device:", DEVICE)

## 2. Configuration
Paper settings are listed alongside faster defaults for a notebook demo. Increase rounds/local epochs only after validating the pipeline.

In [ ]:
CONFIG = {
    "num_clients": 4,
    "communication_rounds": 8,       # paper describes 50 rounds; increase for experiments
    "local_epochs": 1,                # paper uses 5 local epochs
    "batch_size": 32,
    "learning_rate": 5e-4,
    "server_learning_rate": 1e-3,
    "trust_temperature": 0.8,
    "clip_norm": 1.5,
    "dp_epsilon_reference": 1.0,
    "dp_delta": 1e-5,
    "dp_noise_multiplier": 0.7,
    "image_size": 64,                # demo speed; paper parameter table lists 224x224x64
    "ecg_length": 512,                # demo speed; paper lists 4096
    "clinical_features": 56,          # paper parameter table
    "feature_dim": 64,
    "num_heads": 4,
    "num_samples": 640,
    "test_size": 0.20,
    "val_size_from_train": 0.125,     # overall approx 70/10/20 train/validation/test
    "output_dir": "FM_MCAN_HDD_outputs",
}
os.makedirs(CONFIG["output_dir"], exist_ok=True)
with open(os.path.join(CONFIG["output_dir"], "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)
CONFIG

## 3. Data preparation: aligned multimodal demo data
This generator makes reproducible synthetic inputs only to test the full software workflow. Replace it with patient-level aligned data loaders before scientific evaluation. Labels: 0 = no disease, 1 = heart disease.

In [ ]:
def make_synthetic_multimodal(n=640, image_size=64, ecg_length=512, n_clinical=56, seed=42):
    rng = np.random.default_rng(seed)
    y = rng.binomial(1, 0.48, size=n).astype(np.int64)

    # Image-like arrays: label-associated spatial patterns plus noise (not medical images).
    images = rng.normal(0, 0.55, (n, 1, image_size, image_size)).astype(np.float32)
    yy, xx = np.mgrid[-1:1:complex(image_size), -1:1:complex(image_size)]
    lesion = np.exp(-((xx-0.25)**2 + (yy+0.1)**2)/0.08).astype(np.float32)
    for i in range(n):
        images[i, 0] += (0.65 if y[i] else 0.15) * lesion
    images = np.clip(images, -2.5, 2.5)

    # ECG-like sequences: synthetic periodic waveform and label-dependent perturbations.
    t = np.linspace(0, 8*np.pi, ecg_length, dtype=np.float32)
    ecg = np.zeros((n, ecg_length), dtype=np.float32)
    for i in range(n):
        phase = rng.uniform(-0.2, 0.2)
        base = 0.55*np.sin(t + phase) + 0.18*np.sin(3*t + phase)
        abnormal = y[i] * (0.35*np.sin(7*t + phase) + 0.20*rng.normal(size=ecg_length))
        ecg[i] = base + abnormal + rng.normal(0, 0.12, ecg_length)
    ecg = (ecg - ecg.mean(axis=1, keepdims=True)) / (ecg.std(axis=1, keepdims=True) + 1e-6)

    # Clinical tabular features: correlated synthetic values, not real patient records.
    clinical = rng.normal(0, 1, (n, n_clinical)).astype(np.float32)
    clinical[:, :8] += y[:, None] * rng.uniform(0.25, 0.8, (1, 8)).astype(np.float32)
    # Simulate a small amount of missingness for preprocessing demonstration.
    missing = rng.random(clinical.shape) < 0.015
    clinical[missing] = np.nan

    # Additional demo tasks: 0 low / 1 medium / 2 high risk; continuous severity.
    risk = np.clip(y + rng.binomial(1, 0.22, n), 0, 2).astype(np.int64)
    severity = np.clip(0.15 + 0.65*y + rng.normal(0, 0.12, n), 0, 1).astype(np.float32)
    return images, ecg, clinical, y, risk, severity

images, ecg, clinical, labels, risk_labels, severity_targets = make_synthetic_multimodal(
    n=CONFIG["num_samples"], image_size=CONFIG["image_size"],
    ecg_length=CONFIG["ecg_length"], n_clinical=CONFIG["clinical_features"], seed=SEED
)
print("Images:", images.shape, "ECG:", ecg.shape, "clinical:", clinical.shape)
print("Labels:", np.bincount(labels), "| missing clinical values:", int(np.isnan(clinical).sum()))

## 4. Train/validation/test split and local clinical preprocessing
The split is stratified. Fit the imputer/scaler on training data only to avoid test leakage.

In [ ]:
idx = np.arange(len(labels))
trainval_idx, test_idx = train_test_split(
    idx, test_size=CONFIG["test_size"], random_state=SEED, stratify=labels
)
train_idx, val_idx = train_test_split(
    trainval_idx, test_size=CONFIG["val_size_from_train"],
    random_state=SEED, stratify=labels[trainval_idx]
)
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()
clinical_train = imputer.fit_transform(clinical[train_idx])
scaler.fit(clinical_train)
clinical_processed = scaler.transform(imputer.transform(clinical)).astype(np.float32)

print(f"Train={len(train_idx)}, validation={len(val_idx)}, test={len(test_idx)}")
print("Train class distribution:", np.bincount(labels[train_idx]))

## 5. Dataset and simulated hospital partitions
Each client receives its own local subset. For a faithful real deployment, each institution should load its own local files and raw patient records must never be copied to the server.

In [ ]:
class MultiModalDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices)
    def __len__(self): return len(self.indices)
    def __getitem__(self, j):
        i = int(self.indices[j])
        return {
            "image": torch.tensor(images[i], dtype=torch.float32),
            "ecg": torch.tensor(ecg[i], dtype=torch.float32),
            "clinical": torch.tensor(clinical_processed[i], dtype=torch.float32),
            "diagnosis": torch.tensor(labels[i], dtype=torch.long),
            "risk": torch.tensor(risk_labels[i], dtype=torch.long),
            "severity": torch.tensor(severity_targets[i], dtype=torch.float32),
        }

# Partition training indices into four non-overlapping simulated hospital silos.
rng = np.random.default_rng(SEED)
shuffled_train = rng.permutation(train_idx)
client_indices = [part for part in np.array_split(shuffled_train, CONFIG["num_clients"])]
client_loaders = [
    DataLoader(MultiModalDataset(part), batch_size=CONFIG["batch_size"], shuffle=True)
    for part in client_indices
]
val_loader = DataLoader(MultiModalDataset(val_idx), batch_size=CONFIG["batch_size"], shuffle=False)
test_loader = DataLoader(MultiModalDataset(test_idx), batch_size=CONFIG["batch_size"], shuffle=False)
print("Client sample counts:", [len(x) for x in client_indices])

## 6. Model blocks: CVTF-Net-inspired image encoder, TCANet-inspired ECG encoder, TabTransformer and cross-modal attention
The compact blocks below are practical approximations inspired by the paper, not exact reproductions of the paper's unspecified layer-by-layer architectures.

In [ ]:
class ImageEncoder(nn.Module):
    # Compact CNN + channel attention as a runnable CVTF-Net-inspired image pathway.
    def __init__(self, out_dim=64):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.GELU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.GELU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.GELU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )
        self.project = nn.Linear(64, out_dim)
        self.channel_gate = nn.Sequential(nn.Linear(out_dim, out_dim//2), nn.ReLU(),
                                          nn.Linear(out_dim//2, out_dim), nn.Sigmoid())
    def forward(self, x):
        z = self.project(self.features(x))
        return z * self.channel_gate(z)

class ECGEncoder(nn.Module):
    # 1-D convolution + bidirectional recurrent encoder + temporal attention.
    def __init__(self, out_dim=64, hidden=32):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, padding=3), nn.BatchNorm1d(16), nn.GELU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 32, kernel_size=5, padding=2), nn.BatchNorm1d(32), nn.GELU()
        )
        self.rnn = nn.GRU(32, hidden, batch_first=True, bidirectional=True)
        self.attn = nn.Linear(hidden*2, 1)
        self.project = nn.Linear(hidden*2, out_dim)
        self.last_attention = None
    def forward(self, x):
        h = self.conv(x.unsqueeze(1)).transpose(1, 2)
        h, _ = self.rnn(h)
        a = torch.softmax(self.attn(h).squeeze(-1), dim=1)
        self.last_attention = a
        context = torch.sum(h * a.unsqueeze(-1), dim=1)
        return self.project(context)

class TabularEncoder(nn.Module):
    # Feature tokenization and Transformer encoder for structured clinical variables.
    def __init__(self, n_features=56, out_dim=64, nhead=4):
        super().__init__()
        self.token = nn.Linear(1, out_dim)
        layer = nn.TransformerEncoderLayer(d_model=out_dim, nhead=nhead,
                                           dim_feedforward=out_dim*2, dropout=0.1,
                                           batch_first=True, activation="gelu")
        self.transformer = nn.TransformerEncoder(layer, num_layers=2)
        self.norm = nn.LayerNorm(out_dim)
    def forward(self, x):
        tokens = self.token(x.unsqueeze(-1))
        return self.norm(self.transformer(tokens).mean(dim=1))

class CrossModalFusion(nn.Module):
    def __init__(self, dim=64, heads=4):
        super().__init__()
        self.attn_img_ecg = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.attn_img_tab = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.attn_ecg_tab = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.gate = nn.Sequential(nn.Linear(dim*3, dim*3), nn.GELU(), nn.Linear(dim*3, 3), nn.Softmax(dim=-1))
    def forward(self, zi, ze, zt):
        tokens = torch.stack([zi, ze, zt], dim=1)
        # Cross-modal queries: each modality attends to the other two tokens.
        mixed, _ = self.attn_img_ecg(tokens, tokens, tokens, need_weights=False)
        mixed2, _ = self.attn_img_tab(mixed, tokens, tokens, need_weights=False)
        mixed3, _ = self.attn_ecg_tab(mixed2, tokens, tokens, need_weights=False)
        weights = self.gate(torch.cat([zi, ze, zt], dim=-1))
        stacked = torch.stack([mixed3[:,0], mixed3[:,1], mixed3[:,2]], dim=1)
        fused = torch.sum(stacked * weights.unsqueeze(-1), dim=1)
        return self.norm(fused)

class FM_MCAN_HDD(nn.Module):
    def __init__(self, n_clinical=56, dim=64, heads=4):
        super().__init__()
        self.image_encoder = ImageEncoder(dim)
        self.ecg_encoder = ECGEncoder(dim)
        self.tabular_encoder = TabularEncoder(n_clinical, dim, heads)
        self.fusion = CrossModalFusion(dim, heads)
        self.personalization = nn.Sequential(nn.Linear(dim, dim), nn.GELU(), nn.Dropout(0.15))
        # CA-FDH-inspired multi-task outputs: diagnosis, 3-level risk, severity regression.
        self.diagnosis_head = nn.Linear(dim, 2)
        self.risk_head = nn.Linear(dim, 3)
        self.severity_head = nn.Sequential(nn.Linear(dim, 1), nn.Sigmoid())
    def forward(self, image, ecg, clinical):
        zi = self.image_encoder(image)
        ze = self.ecg_encoder(ecg)
        zt = self.tabular_encoder(clinical)
        fused = self.fusion(zi, ze, zt)
        h = self.personalization(fused)
        return {
            "diagnosis": self.diagnosis_head(h),
            "risk": self.risk_head(h),
            "severity": self.severity_head(h).squeeze(-1),
            "embeddings": (zi, ze, zt, fused),
        }

model = FM_MCAN_HDD(n_clinical=CONFIG["clinical_features"], dim=CONFIG["feature_dim"],
                    heads=CONFIG["num_heads"]).to(DEVICE)
print("Model parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

## 7. Local training and validation utilities
Diagnosis is the primary objective; risk and severity are auxiliary tasks. Client weights are updated locally, while the server only receives model parameters in this simulation.

In [ ]:
def move_batch(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}

def multitask_loss(out, batch):
    loss_diag = F.cross_entropy(out["diagnosis"], batch["diagnosis"])
    loss_risk = F.cross_entropy(out["risk"], batch["risk"])
    loss_severity = F.smooth_l1_loss(out["severity"], batch["severity"])
    return loss_diag + 0.20*loss_risk + 0.15*loss_severity

def train_local(global_state, loader, epochs=1):
    local_model = FM_MCAN_HDD(n_clinical=CONFIG["clinical_features"],
                              dim=CONFIG["feature_dim"], heads=CONFIG["num_heads"]).to(DEVICE)
    local_model.load_state_dict(global_state)
    local_model.train()
    optimizer = torch.optim.AdamW(local_model.parameters(), lr=CONFIG["learning_rate"], weight_decay=1e-4)
    losses = []
    for _ in range(epochs):
        for batch in loader:
            batch = move_batch(batch)
            optimizer.zero_grad(set_to_none=True)
            out = local_model(batch["image"], batch["ecg"], batch["clinical"])
            loss = multitask_loss(out, batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(local_model.parameters(), max_norm=CONFIG["clip_norm"])
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
    return {k: v.detach().cpu().clone() for k, v in local_model.state_dict().items()}, float(np.mean(losses) if losses else np.nan)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    y_true, y_prob, y_pred, losses = [], [], [], []
    risk_true, risk_pred = [], []
    for batch in loader:
        batch = move_batch(batch)
        out = model(batch["image"], batch["ecg"], batch["clinical"])
        losses.append(float(multitask_loss(out, batch).cpu()))
        probs = torch.softmax(out["diagnosis"], dim=1)[:, 1]
        y_prob.extend(probs.cpu().numpy())
        y_pred.extend(out["diagnosis"].argmax(1).cpu().numpy())
        y_true.extend(batch["diagnosis"].cpu().numpy())
        risk_true.extend(batch["risk"].cpu().numpy())
        risk_pred.extend(out["risk"].argmax(1).cpu().numpy())
    metrics = {
        "loss": float(np.mean(losses)),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float("nan"),
    }
    return metrics, np.asarray(y_true), np.asarray(y_pred), np.asarray(y_prob), (np.asarray(risk_true), np.asarray(risk_pred))

## 8. Trust-aware federated aggregation with clipping and Gaussian noise
Trust scores combine local training-set size with local validation quality. In this prototype, all clients share the same aligned architecture for demonstration. Noise is applied to the aggregated parameter delta. The epsilon/delta settings are reference parameters only: formal DP accounting is not implemented.

In [ ]:
def softmax_np(x):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x)
    e = np.exp(x)
    return e / (e.sum() + 1e-12)

def aggregate_trust_dp(global_state, client_states, trust_scores, client_sizes):
    # Temperature-scaled trust weights, then incorporate client sample counts.
    trust_logits = np.asarray(trust_scores, dtype=np.float64) / CONFIG["trust_temperature"]
    trust_weights = softmax_np(trust_logits)
    size_weights = np.asarray(client_sizes, dtype=np.float64)
    size_weights = size_weights / size_weights.sum()
    weights = trust_weights * size_weights
    weights = weights / weights.sum()

    new_state = {}
    for key, global_tensor in global_state.items():
        if not torch.is_floating_point(global_tensor):
            new_state[key] = global_tensor.clone()
            continue
        deltas = []
        for local_state in client_states:
            deltas.append(local_state[key].float() - global_tensor.float())
        # Weighted aggregate delta.
        delta = sum(float(w) * d for w, d in zip(weights, deltas))
        # Global norm clipping of the aggregate update.
        norm = torch.linalg.vector_norm(delta)
        if norm > CONFIG["clip_norm"]:
            delta = delta * (CONFIG["clip_norm"] / (norm + 1e-12))
        # Approximate Gaussian mechanism; not a formal DP implementation.
        sigma = CONFIG["dp_noise_multiplier"] * CONFIG["clip_norm"]
        delta = delta + torch.randn_like(delta) * sigma / max(1, len(client_states))
        new_state[key] = global_tensor.float() + CONFIG["server_learning_rate"] * delta
    return new_state, weights

@torch.no_grad()
def score_client(global_state, loader):
    local_model = FM_MCAN_HDD(n_clinical=CONFIG["clinical_features"],
                              dim=CONFIG["feature_dim"], heads=CONFIG["num_heads"]).to(DEVICE)
    local_model.load_state_dict(global_state)
    metrics, *_ = evaluate(local_model, loader)
    return float(metrics["accuracy"])

## 9. Federated training loop
The client data remain in local DataLoaders; only model states and summary trust scores are used by the server code.

In [ ]:
global_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
history = []
start_time = time.time()

for rnd in range(1, CONFIG["communication_rounds"] + 1):
    client_states, local_losses, trust_scores, sizes = [], [], [], []
    # Broadcast current global state, train each simulated hospital locally.
    for client_id, loader in enumerate(client_loaders):
        state_k, loss_k = train_local(global_state, loader, epochs=CONFIG["local_epochs"])
        client_states.append(state_k)
        local_losses.append(loss_k)
        sizes.append(len(loader.dataset))
        # Trust score = current global model validation quality on this client's local subset.
        # For a real deployment, validation must be computed locally and only a permitted summary shared.
        trust_scores.append(score_client(global_state, loader))
    global_state, agg_weights = aggregate_trust_dp(global_state, client_states, trust_scores, sizes)
    model.load_state_dict(global_state)
    val_metrics, *_ = evaluate(model, val_loader)
    row = {"round": rnd, "train_loss": float(np.mean(local_losses)),
           **{f"val_{k}": v for k, v in val_metrics.items()},
           "trust_mean": float(np.mean(trust_scores))}
    history.append(row)
    print(f"Round {rnd:02d}/{CONFIG['communication_rounds']} | "
          f"local loss={row['train_loss']:.4f} | val acc={val_metrics['accuracy']:.3f} | "
          f"val AUC={val_metrics['auc']:.3f} | weights={np.round(agg_weights, 3)}")

history_df = pd.DataFrame(history)
history_df.to_csv(os.path.join(CONFIG["output_dir"], "federated_training_history.csv"), index=False)
print(f"Training elapsed: {(time.time()-start_time)/60:.2f} minutes")

## 10. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history_df["round"], history_df["val_accuracy"], marker="o", label="Validation accuracy")
axes[0].set_xlabel("Communication round"); axes[0].set_ylabel("Accuracy"); axes[0].set_ylim(0, 1)
axes[0].grid(alpha=0.3); axes[0].legend()
axes[1].plot(history_df["round"], history_df["train_loss"], marker="o", label="Mean local loss")
axes[1].plot(history_df["round"], history_df["val_loss"], marker="s", label="Validation loss")
axes[1].set_xlabel("Communication round"); axes[1].set_ylabel("Loss")
axes[1].grid(alpha=0.3); axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "training_curves.png"), dpi=220, bbox_inches="tight")
plt.show()

## 11. Final held-out test evaluation
These metrics describe only the synthetic demonstration generated above.

In [ ]:
test_metrics, y_true, y_pred, y_prob, risk_eval = evaluate(model, test_loader)
print("Synthetic demo test metrics:")
print(pd.Series(test_metrics).round(4))
print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["No disease", "Heart disease"], zero_division=0))

cm = confusion_matrix(y_true, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No disease", "Heart disease"],
            yticklabels=["No disease", "Heart disease"], ax=axes[0])
axes[0].set_title("Confusion matrix"); axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
fpr, tpr, _ = roc_curve(y_true, y_prob)
axes[1].plot(fpr, tpr, label=f"AUC = {test_metrics['auc']:.3f}")
axes[1].plot([0,1], [0,1], linestyle="--")
axes[1].set_xlabel("False positive rate"); axes[1].set_ylabel("True positive rate")
axes[1].set_title("ROC curve"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "test_evaluation.png"), dpi=220, bbox_inches="tight")
plt.show()

with open(os.path.join(CONFIG["output_dir"], "test_metrics.json"), "w") as f:
    json.dump({k: float(v) for k,v in test_metrics.items()}, f, indent=2)

## 12. Explainability: image saliency, ECG temporal attention, clinical feature sensitivity
These are local, gradient/attention-based inspection examples. They are not clinical explanations validated by a cardiologist. No attribution is sent to a server.

In [ ]:
# Select one test example and compute gradients locally.
sample = next(iter(test_loader))
sample = move_batch(sample)
model.eval()
img = sample["image"][:1].detach().clone().requires_grad_(True)
sig = sample["ecg"][:1].detach().clone().requires_grad_(True)
clin = sample["clinical"][:1].detach().clone().requires_grad_(True)
out = model(img, sig, clin)
pred_class = int(out["diagnosis"].argmax(1).item())
score = out["diagnosis"][0, pred_class]
model.zero_grad(set_to_none=True)
score.backward()

# Image gradient saliency map (portable fallback; Grad-CAM-style localization needs target-layer hooks).
img_saliency = img.grad.detach().abs().mean(dim=1)[0].cpu().numpy()
img_saliency = img_saliency / (img_saliency.max() + 1e-8)

# ECG temporal attention from the TCANet-inspired encoder.
with torch.no_grad():
    _ = model(sample["image"][:1], sample["ecg"][:1], sample["clinical"][:1])
    attn = model.ecg_encoder.last_attention[0].detach().cpu().numpy()

# Clinical feature sensitivity: absolute gradient magnitude.
clinical_sensitivity = clin.grad.detach().abs()[0].cpu().numpy()
top_k = np.argsort(clinical_sensitivity)[-10:][::-1]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(sample["image"][0,0].detach().cpu(), cmap="gray")
axes[0].imshow(img_saliency, cmap="jet", alpha=0.45)
axes[0].set_title(f"Image saliency (pred={pred_class})"); axes[0].axis("off")
axes[1].plot(attn); axes[1].set_title("ECG temporal attention")
axes[1].set_xlabel("Encoded time step"); axes[1].set_ylabel("Attention weight"); axes[1].grid(alpha=0.25)
axes[2].barh([f"Feature {i}" for i in top_k[::-1]], clinical_sensitivity[top_k[::-1]])
axes[2].set_title("Top clinical feature sensitivities"); axes[2].set_xlabel("|gradient|")
plt.tight_layout()
plt.savefig(os.path.join(CONFIG["output_dir"], "local_explainability.png"), dpi=220, bbox_inches="tight")
plt.show()

## 13. Local personalization (C³A-FusionNet-inspired refinement)
A simple local fine-tuning pass illustrates the local-refinement stage after global aggregation. This produces a client-specific model and does not overwrite the global model.

In [ ]:
def personalize_local(global_state, loader, epochs=1):
    personalized = FM_MCAN_HDD(n_clinical=CONFIG["clinical_features"],
                                dim=CONFIG["feature_dim"], heads=CONFIG["num_heads"]).to(DEVICE)
    personalized.load_state_dict(global_state)
    personalized.train()
    optimizer = torch.optim.AdamW(personalized.parameters(), lr=CONFIG["learning_rate"]*0.25)
    for _ in range(epochs):
        for batch in loader:
            batch = move_batch(batch)
            optimizer.zero_grad(set_to_none=True)
            out = personalized(batch["image"], batch["ecg"], batch["clinical"])
            loss = multitask_loss(out, batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(personalized.parameters(), CONFIG["clip_norm"])
            optimizer.step()
    return personalized

personalized_client0 = personalize_local(global_state, client_loaders[0], epochs=1)
personalized_metrics, *_ = evaluate(personalized_client0, val_loader)
print("Client-0 personalized model validation metrics (synthetic demo):")
print(pd.Series(personalized_metrics).round(4))

## 14. Baseline and ablation-ready comparisons
This cell runs a FedAvg aggregation baseline using the same local updates and without trust weighting/noise, so the comparison isolates aggregation behavior in this compact prototype. It is not a recreation of all published baseline methods.

In [ ]:
def aggregate_fedavg(global_state, client_states, client_sizes):
    weights = np.asarray(client_sizes, dtype=float)
    weights /= weights.sum()
    out = {}
    for key, base in global_state.items():
        if torch.is_floating_point(base):
            out[key] = sum(float(w) * state[key].float() for w, state in zip(weights, client_states))
        else:
            out[key] = base.clone()
    return out

# One-round FedAvg comparator from the same starting global state.
fedavg_states, fedavg_sizes = [], []
for loader in client_loaders:
    state, _ = train_local(global_state, loader, epochs=1)
    fedavg_states.append(state); fedavg_sizes.append(len(loader.dataset))
fedavg_state = aggregate_fedavg(global_state, fedavg_states, fedavg_sizes)
fedavg_model = FM_MCAN_HDD(n_clinical=CONFIG["clinical_features"],
                           dim=CONFIG["feature_dim"], heads=CONFIG["num_heads"]).to(DEVICE)
fedavg_model.load_state_dict(fedavg_state)
fedavg_metrics, *_ = evaluate(fedavg_model, test_loader)

comparison = pd.DataFrame([
    {"Method": "FM-MCAN-HDD prototype (trust + noise)", **test_metrics},
    {"Method": "FedAvg comparator (one local round)", **fedavg_metrics},
])
comparison.to_csv(os.path.join(CONFIG["output_dir"], "aggregation_comparison.csv"), index=False)
display(comparison.round(4))

## 15. Optional stratified 5-fold evaluation helper
Cross-validation requires retraining from scratch inside each fold. This helper demonstrates split construction and class balance; full 5-fold deep federated retraining is intentionally left as an opt-in because it multiplies training time. Do not treat the precomputed metrics in the paper as outputs from this notebook.

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_summary = []
for fold, (tr, te) in enumerate(skf.split(np.zeros(len(labels)), labels), start=1):
    fold_summary.append({
        "fold": fold, "train_samples": len(tr), "test_samples": len(te),
        "train_positive_rate": float(labels[tr].mean()),
        "test_positive_rate": float(labels[te].mean())
    })
fold_df = pd.DataFrame(fold_summary)
fold_df.to_csv(os.path.join(CONFIG["output_dir"], "five_fold_split_summary.csv"), index=False)
display(fold_df)

## 16. Save final model and all outputs

In [ ]:
model_path = os.path.join(CONFIG["output_dir"], "fm_mcan_hdd_global_model.pt")
torch.save({
    "model_state_dict": model.state_dict(),
    "config": CONFIG,
    "class_names": ["No disease", "Heart disease"],
    "note": "Synthetic-data research prototype; not a clinical model."
}, model_path)

print("Saved artifacts:")
for p in sorted(Path(CONFIG["output_dir"]).glob("*")):
    print(" -", p.resolve(), f"({p.stat().st_size:,} bytes)")

## 17. Adapting this notebook to real datasets (required before claiming research results)

1. **Cardiac CT/MRI:** load images/volumes from the dataset described in the paper; keep patient IDs and study IDs. Split by patient before creating slices to prevent leakage. The demo uses 2-D single-channel images; true 3-D CVTF-Net requires a 3-D encoder and suitable GPU memory.
2. **ECG chart images:** use the labelled ECG-image dataset described in the manuscript, with image preprocessing (anisotropic diffusion / CLAHE as appropriate) and patient-level splitting.
3. **Raw ECG:** load CSV/EDF/WFDB signals and their labels; verify sampling rate, lead layout and label mapping. Apply the paper's 0.5–40 Hz filtering only when appropriate to the data.
4. **Clinical records:** map actual columns to the intended target, fit imputation/scaling only on each training partition, and document missingness.
5. **Multimodal alignment:** join modalities only with a verified patient/study key and a clearly defined label. The paper's source datasets are different sizes and are not automatically patient-aligned. Do not fabricate cross-modal patient matches.
6. **Hospital partition:** create client datasets locally. For heterogeneous single-modality sites, implement modality-specific local models and aggregate only a compatible shared representation/head; do not average incompatible model tensors.
7. **Security:** use a reviewed secure aggregation protocol, authenticated TLS, key management and a vetted privacy accountant before claiming encryption or an ε-DP guarantee. This notebook's Gaussian-noise simulation is not sufficient.
8. **Evaluation:** report metrics from held-out real patients, patient-level 5-fold cross-validation, confidence intervals, and external validation. Do not hard-code manuscript metrics into model outputs.

### Paper-to-code coverage
- **Local preprocessing:** synthetic image/ECG/tabular preparation and train-only clinical imputation/scaling; real imaging registration, Attention-UNet++, and exact Pan–Tompkins preprocessing require dataset-specific implementation.
- **CVTF-Net / TCANet / TabTransformer:** compact runnable inspired encoders included.
- **GF-MCAN-TS:** trust-weighted aggregation, update clipping, noise simulation and cross-modal attention included; true Paillier homomorphic aggregation and formal DP accountant are not implemented.
- **C³A-FusionNet:** local fine-tuning/personalization included; full contrastive objective is not implemented.
- **CA-FDH:** diagnosis, 3-level risk and severity outputs included.
- **Explainability:** image saliency, ECG attention and clinical gradients included; Grad-CAM hooks, SHAP and concept-bottleneck rules can be added after real data and model validation.
- **Reproducibility:** config, metrics, plots, model checkpoint and CSV summaries are saved under `FM_MCAN_HDD_outputs/`.

**Clinical safety:** this notebook is for software research and education only. It must not be used to diagnose or treat patients.